# DLinear


## Imports


In [ ]:
import os, sys, warnings, math
warnings.filterwarnings('ignore')
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.pipeline import Pipeline

import mlflow
import mlflow.pyfunc
import mlflow.sklearn
import dagshub
from src.wmae import wmae
from src.cv_split import time_series_cv_splits
from src.features import load_raw_data, merge_all, build_calendar_features, build_markdown_features, build_store_features, TARGET

SEED      = 42
DATA_DIR  = '.'
DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

torch.manual_seed(SEED)
np.random.seed(SEED)

print(f'PyTorch {torch.__version__} | Device: {DEVICE}')
print('All imports successful')


## MLflow Setup


In [ ]:
os.environ.setdefault('MLFLOW_TRACKING_USERNAME', 'sansi23')

dagshub.init(
    repo_owner='sansi23',
    repo_name='Walmart-Recruiting---Store-Sales-Forecasting',
    mlflow=True
)
dlinear_experiment = mlflow.set_experiment('DLinear_Training')
print('MLflow experiment:', dlinear_experiment.name)
print('Tracking URI:', mlflow.get_tracking_uri())

## Runtime Setup


In [ ]:
import copy
import time
import zipfile

if 'train_raw' not in globals() or 'test_raw' not in globals():
    train_raw, test_raw, features_raw, stores_raw = load_raw_data(DATA_DIR)
    train_raw, test_raw = merge_all(train_raw, test_raw, features_raw, stores_raw)

DLINEAR_DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DLINEAR_PIN_MEMORY = DLINEAR_DEVICE.type == 'cuda'
DLINEAR_BATCH_SIZE = 2048 if DLINEAR_DEVICE.type == 'cuda' else 512
DLINEAR_NUM_WORKERS = 0

if DLINEAR_DEVICE.type == 'cuda':
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
    if hasattr(torch, 'set_float32_matmul_precision'):
        torch.set_float32_matmul_precision('high')
    print('CUDA is available')
    print('Device name:', torch.cuda.get_device_name(0))
else:
    print('CUDA is NOT available in this Python environment; training will run on CPU.')

print('Device:', DLINEAR_DEVICE)
print('Batch size:', DLINEAR_BATCH_SIZE)


## Data Preparation


In [ ]:
def prepare_dlinear_frame(df):
    out = df.copy()
    out['Date'] = pd.to_datetime(out['Date'])
    out = out.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)
    if TARGET in out.columns:
        out[TARGET] = out[TARGET].clip(lower=0.0)
    for col in ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']:
        if col in out.columns:
            out[col] = out[col].fillna(0.0).clip(lower=0.0)
    for col in ['CPI', 'Unemployment', 'Temperature', 'Fuel_Price']:
        if col in out.columns:
            out[col] = out.groupby('Store')[col].transform(lambda s: s.ffill().bfill())

    week = out['Date'].dt.isocalendar().week.astype(float)
    angle = 2 * np.pi * week / 52.0
    out['WeekSin'] = np.sin(angle)
    out['WeekCos'] = np.cos(angle)
    out['WeekSin2'] = np.sin(2 * angle)
    out['WeekCos2'] = np.cos(2 * angle)

    is_holiday = out['IsHoliday'].fillna(False).astype(bool)
    month = out['Date'].dt.month
    out['IsSuperBowl'] = (is_holiday & month.eq(2)).astype(float)
    out['IsLaborDay'] = (is_holiday & month.eq(9)).astype(float)
    out['IsThanksgiving'] = (is_holiday & month.eq(11)).astype(float)
    out['IsChristmas'] = (is_holiday & month.eq(12)).astype(float)

    markdown_cols = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']
    markdown_total = out[markdown_cols].sum(axis=1)
    out['MarkDownLog'] = np.log1p(markdown_total) / 10.0
    out['MarkDownCountScaled'] = out[markdown_cols].gt(0).sum(axis=1) / 5.0
    return out

DLINEAR_FUTURE_FEATURES = [
    'WeekSin', 'WeekCos', 'WeekSin2', 'WeekCos2',
    'IsHoliday', 'IsSuperBowl', 'IsLaborDay', 'IsThanksgiving', 'IsChristmas',
    'MarkDownLog', 'MarkDownCountScaled',
]

DLINEAR_LOOKBACK = 52
DLINEAR_HORIZON = 39
DLINEAR_VAL_CUTOFF = pd.Timestamp('2012-01-27')
DLINEAR_VAL_END = pd.Timestamp('2012-10-26')
DLINEAR_TEST_ORIGIN = pd.to_datetime(train_raw['Date']).max()

dlinear_train = prepare_dlinear_frame(train_raw)
dlinear_test = prepare_dlinear_frame(test_raw)

dlinear_train_history = dlinear_train[dlinear_train['Date'] <= DLINEAR_VAL_CUTOFF].copy()
dlinear_val = dlinear_train[
    (dlinear_train['Date'] > DLINEAR_VAL_CUTOFF) &
    (dlinear_train['Date'] <= DLINEAR_VAL_END)
].copy()

print('Train history:', dlinear_train_history['Date'].min(), '->', dlinear_train_history['Date'].max(), dlinear_train_history.shape)
print('Validation:', dlinear_val['Date'].min(), '->', dlinear_val['Date'].max(), dlinear_val.shape)
print('Final train:', dlinear_train['Date'].min(), '->', dlinear_train['Date'].max(), dlinear_train.shape)
print('Test:', dlinear_test['Date'].min(), '->', dlinear_test['Date'].max(), dlinear_test.shape)


## Model and Dataset


In [ ]:
class CleanSeriesDecomposition(nn.Module):
    def __init__(self, kernel_size=25):
        super().__init__()
        pad = kernel_size // 2
        self.pad = nn.ReplicationPad1d((pad, pad))
        self.avg = nn.AvgPool1d(kernel_size=kernel_size, stride=1, padding=0)

    def forward(self, x):
        x_3d = x.unsqueeze(1)
        trend = self.avg(self.pad(x_3d)).squeeze(1)
        seasonal = x - trend
        return trend, seasonal


class CleanDLinear(nn.Module):
    def __init__(self, lookback, horizon, kernel_size=25, future_feature_count=0):
        super().__init__()
        self.decomp = CleanSeriesDecomposition(kernel_size)
        self.trend_layer = nn.Linear(lookback, horizon)
        self.seasonal_layer = nn.Linear(lookback, horizon)
        self.future_layer = nn.Linear(future_feature_count, 1) if future_feature_count else None
        nn.init.xavier_uniform_(self.trend_layer.weight)
        nn.init.xavier_uniform_(self.seasonal_layer.weight)

    def forward(self, x, future_features=None):
        trend, seasonal = self.decomp(x)
        forecast = self.trend_layer(trend) + self.seasonal_layer(seasonal)
        if self.future_layer is not None:
            forecast = forecast + self.future_layer(future_features).squeeze(-1)
        return forecast


def dlinear_norm_stats(df):
    stats = (df.groupby(['Store', 'Dept'])[TARGET]
               .agg(['mean', 'std'])
               .rename(columns={'mean': 'series_mean', 'std': 'series_std'}))
    stats['series_std'] = stats['series_std'].fillna(1.0).replace(0, 1.0)
    return stats


class DLinearWindowDataset(Dataset):
    def __init__(self, df, lookback, horizon, norm_stats, future_feature_cols):
        self.samples = []
        self.lookback = lookback
        self.horizon = horizon
        self.future_feature_cols = future_feature_cols

        for (store, dept), group in df.sort_values('Date').groupby(['Store', 'Dept']):
            if (store, dept) not in norm_stats.index:
                continue
            sales = group[TARGET].to_numpy(dtype=np.float32)
            holiday_weight = np.where(group['IsHoliday'].to_numpy(dtype=bool), 5.0, 1.0).astype(np.float32)
            mean = np.float32(norm_stats.loc[(store, dept), 'series_mean'])
            std = np.float32(norm_stats.loc[(store, dept), 'series_std'])
            sales_norm = (sales - mean) / std
            future = group[future_feature_cols].to_numpy(dtype=np.float32)

            n = len(sales_norm) - lookback - horizon + 1
            for i in range(max(0, n)):
                x = sales_norm[i:i + lookback]
                y = sales_norm[i + lookback:i + lookback + horizon]
                w = holiday_weight[i + lookback:i + lookback + horizon]
                f = future[i + lookback:i + lookback + horizon]
                self.samples.append((x, y, w, f))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        x, y, w, f = self.samples[idx]
        return torch.from_numpy(x), torch.from_numpy(y), torch.from_numpy(w), torch.from_numpy(f)


def weighted_l1_loss(pred, y_true, weights):
    return (torch.abs(pred - y_true) * weights).sum() / weights.sum().clamp_min(1.0)


def train_one_dlinear_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    total_rows = 0
    for x, y_true, weights, future_features in loader:
        x = x.to(device, non_blocking=True).float()
        y_true = y_true.to(device, non_blocking=True).float()
        weights = weights.to(device, non_blocking=True).float()
        future_features = future_features.to(device, non_blocking=True).float()

        optimizer.zero_grad(set_to_none=True)
        pred = model(x, future_features)
        loss = weighted_l1_loss(pred, y_true, weights)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item() * len(x)
        total_rows += len(x)
    return total_loss / max(total_rows, 1)


## Prediction and Validation


In [ ]:
def predict_dlinear_origin(model, history_df, target_df, norm_stats, lookback, horizon, origin_date, device, future_feature_cols, batch_size=None, pin_memory=None):
    model.eval()
    origin_date = pd.Timestamp(origin_date)
    batch_size = batch_size or DLINEAR_BATCH_SIZE
    pin_memory = DLINEAR_PIN_MEMORY if pin_memory is None else pin_memory
    history_df = history_df.sort_values(['Store', 'Dept', 'Date'])
    target_df = target_df.copy().reset_index(drop=True)
    history_groups = {
        key: group for key, group in history_df.groupby(['Store', 'Dept'], sort=False)
    }

    series_inputs = []
    series_future = []
    series_meta = []
    for (store, dept), target_group in target_df.groupby(['Store', 'Dept'], sort=False):
        key = (store, dept)
        hist = history_groups.get(key)
        if hist is None or len(hist) < lookback or key not in norm_stats.index:
            continue
        hist = hist[hist['Date'] <= origin_date]
        if len(hist) < lookback:
            continue
        mean = float(norm_stats.loc[key, 'series_mean'])
        std = float(norm_stats.loc[key, 'series_std'])
        x_raw = hist[TARGET].tail(lookback).to_numpy(dtype=np.float32)
        future_block = np.zeros((horizon, len(future_feature_cols)), dtype=np.float32)
        target_group = target_group.sort_values('Date')
        horizon_index = ((target_group['Date'] - origin_date).dt.days // 7 - 1).astype(int).to_numpy()
        valid = (horizon_index >= 0) & (horizon_index < horizon)
        future_block[horizon_index[valid]] = target_group.loc[valid, future_feature_cols].to_numpy(dtype=np.float32)
        series_inputs.append((x_raw - mean) / std)
        series_future.append(future_block)
        series_meta.append((store, dept, mean, std))

    forecast_rows = []
    if series_inputs:
        x_all = torch.from_numpy(np.stack(series_inputs).astype(np.float32))
        future_all = torch.from_numpy(np.stack(series_future).astype(np.float32))
        loader = DataLoader(
            torch.utils.data.TensorDataset(x_all, future_all),
            batch_size=batch_size,
            shuffle=False,
            num_workers=0,
            pin_memory=pin_memory,
        )
        raw_preds = []
        with torch.no_grad():
            for xb, future_b in loader:
                pred_norm = model(
                    xb.to(device, non_blocking=True).float(),
                    future_b.to(device, non_blocking=True).float(),
                ).cpu().numpy()
                raw_preds.append(pred_norm)
        raw_preds = np.vstack(raw_preds)

        for i, (store, dept, mean, std) in enumerate(series_meta):
            pred_raw = raw_preds[i] * std + mean
            forecast_rows.extend(
                (store, dept, horizon_idx, float(value))
                for horizon_idx, value in enumerate(pred_raw)
            )

    forecast_lookup = pd.DataFrame(
        forecast_rows,
        columns=['Store', 'Dept', '_Horizon_Index', '_DLinear_Pred'],
    )

    work = target_df[['Store', 'Dept', 'Date']].copy()
    work['_Row_Order'] = np.arange(len(work))
    work['_Horizon_Index'] = ((work['Date'] - origin_date).dt.days // 7 - 1).astype(int)
    work = work.merge(
        forecast_lookup,
        on=['Store', 'Dept', '_Horizon_Index'],
        how='left',
        sort=False,
    )

    lag_lookup = (
        history_df[['Store', 'Dept', 'Date', TARGET]]
        .drop_duplicates(['Store', 'Dept', 'Date'], keep='last')
        .rename(columns={'Date': '_Lag_Date', TARGET: '_Lag52'})
    )
    work['_Lag_Date'] = work['Date'] - pd.Timedelta(weeks=52)
    work = work.merge(lag_lookup, on=['Store', 'Dept', '_Lag_Date'], how='left', sort=False)

    mean_lookup = (
        history_df.groupby(['Store', 'Dept'], as_index=False)[TARGET]
        .mean()
        .rename(columns={TARGET: '_Series_Mean'})
    )
    work = work.merge(mean_lookup, on=['Store', 'Dept'], how='left', sort=False)

    fallback_mask = work['_DLinear_Pred'].isna()
    global_mean = float(history_df[TARGET].mean())
    work.loc[fallback_mask, '_DLinear_Pred'] = (
        work.loc[fallback_mask, '_Lag52']
        .fillna(work.loc[fallback_mask, '_Series_Mean'])
        .fillna(global_mean)
    )

    work = work.sort_values('_Row_Order')
    preds = work['_DLinear_Pred'].to_numpy(dtype=np.float32)
    return np.clip(preds, 0, None), int(fallback_mask.sum())

def fit_dlinear_with_validation(config, train_history, val_df, val_cutoff, seed=SEED):
    torch.manual_seed(seed)
    if DLINEAR_DEVICE.type == 'cuda':
        torch.cuda.manual_seed_all(seed)

    lookback = int(config['lookback'])
    horizon = int(config['horizon'])
    kernel = int(config['kernel'])
    lr = float(config['lr'])
    weight_decay = float(config.get('weight_decay', 1e-4))
    epochs = int(config.get('epochs', 80))
    patience = int(config.get('patience', 12))

    stats = dlinear_norm_stats(train_history)
    ds = DLinearWindowDataset(train_history, lookback, horizon, stats, DLINEAR_FUTURE_FEATURES)
    if len(ds) == 0:
        raise ValueError('No DLinear training windows were created. Lower lookback/horizon.')

    loader = DataLoader(
        ds,
        batch_size=DLINEAR_BATCH_SIZE,
        shuffle=True,
        num_workers=DLINEAR_NUM_WORKERS,
        pin_memory=DLINEAR_PIN_MEMORY,
    )

    future_feature_count = len(DLINEAR_FUTURE_FEATURES) if config.get('use_future_features', True) else 0
    model = CleanDLinear(
        lookback,
        horizon,
        kernel_size=kernel,
        future_feature_count=future_feature_count,
    ).to(DLINEAR_DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_state = None
    best_wmae = np.inf
    best_epoch = 0
    wait = 0
    history_rows = []
    started = time.time()

    for epoch in range(1, epochs + 1):
        train_loss = train_one_dlinear_epoch(model, loader, optimizer, DLINEAR_DEVICE)
        val_preds, fallback_count = predict_dlinear_origin(
            model, train_history, val_df, stats, lookback, horizon, val_cutoff,
            DLINEAR_DEVICE, DLINEAR_FUTURE_FEATURES
        )
        val_wmae = wmae(val_df[TARGET].values, val_preds, val_df['IsHoliday'].values)
        history_rows.append({'epoch': epoch, 'train_loss': train_loss, 'val_wmae': val_wmae, 'fallback_rows': fallback_count})

        if val_wmae < best_wmae:
            best_wmae = val_wmae
            best_epoch = epoch
            wait = 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            wait += 1

        if epoch == 1 or epoch % 5 == 0:
            print(f"epoch={epoch:03d} train_loss={train_loss:.5f} val_wmae={val_wmae:,.2f} fallback_rows={fallback_count}")

        if wait >= patience:
            print(f'Early stopping at epoch {epoch}; best epoch {best_epoch}')
            break

    model.load_state_dict(best_state)
    elapsed = time.time() - started
    return model, stats, pd.DataFrame(history_rows), best_wmae, best_epoch, elapsed


## Hyperparameter Tuning


In [ ]:
dlinear_search_space = [
    {'lookback': 52, 'horizon': DLINEAR_HORIZON, 'kernel': 25, 'lr': 1e-3, 'weight_decay': 1e-4, 'epochs': 80, 'patience': 12, 'use_future_features': False},
    {'lookback': 52, 'horizon': DLINEAR_HORIZON, 'kernel': 13, 'lr': 1e-3, 'weight_decay': 1e-4, 'epochs': 80, 'patience': 12, 'use_future_features': True},
    {'lookback': 52, 'horizon': DLINEAR_HORIZON, 'kernel': 25, 'lr': 1e-3, 'weight_decay': 1e-4, 'epochs': 80, 'patience': 12, 'use_future_features': True},
    {'lookback': 52, 'horizon': DLINEAR_HORIZON, 'kernel': 25, 'lr': 5e-4, 'weight_decay': 1e-4, 'epochs': 100, 'patience': 15, 'use_future_features': True},
]

dlinear_results = []
dlinear_best = {'wmae': np.inf}

with mlflow.start_run(run_name='DLinear_Tuning'):
    mlflow.log_params({
        'device': str(DLINEAR_DEVICE),
        'batch_size': DLINEAR_BATCH_SIZE,
        'val_cutoff': str(DLINEAR_VAL_CUTOFF.date()),
        'val_horizon_weeks': DLINEAR_HORIZON,
        'holiday_weight': 5,
        'validation_metric': 'WMAE_raw_scale',
        'future_features': ','.join(DLINEAR_FUTURE_FEATURES),
    })

    for trial_id, cfg in enumerate(dlinear_search_space, start=1):
        feature_tag = 'calendar' if cfg['use_future_features'] else 'sales_only'
        run_name = f"DLinear_trial_{trial_id:02d}_{feature_tag}_K{cfg['kernel']}_lr{cfg['lr']}"
        with mlflow.start_run(run_name=run_name, nested=True):
            mlflow.log_params(cfg)
            model_trial, stats_trial, curve_df, best_wmae, best_epoch, elapsed = fit_dlinear_with_validation(
                cfg, dlinear_train_history, dlinear_val, DLINEAR_VAL_CUTOFF
            )
            mlflow.log_metric('val_wmae', float(best_wmae))
            mlflow.log_metric('best_epoch', int(best_epoch))
            mlflow.log_metric('elapsed_seconds', float(elapsed))

            curve_path = f'dlinear_trial_{trial_id:02d}_curve.csv'
            curve_df.to_csv(curve_path, index=False)
            mlflow.log_artifact(curve_path)

        result = {**cfg, 'trial_id': trial_id, 'val_wmae': best_wmae, 'best_epoch': best_epoch, 'elapsed_seconds': elapsed}
        dlinear_results.append(result)
        print(f"trial={trial_id:02d} lookback={cfg['lookback']} kernel={cfg['kernel']} lr={cfg['lr']} -> WMAE={best_wmae:,.2f}")

        if best_wmae < dlinear_best['wmae']:
            dlinear_best = {
                'wmae': best_wmae,
                'config': cfg.copy(),
                'model_state': copy.deepcopy(model_trial.state_dict()),
                'stats': stats_trial.copy(),
            }

    dlinear_tuning_df = pd.DataFrame(dlinear_results).sort_values('val_wmae')
    dlinear_tuning_df.to_csv('dlinear_tuning_results.csv', index=False)
    mlflow.log_artifact('dlinear_tuning_results.csv')
    mlflow.log_metric('best_val_wmae', float(dlinear_best['wmae']))
    mlflow.log_params({f"best_{k}": v for k, v in dlinear_best['config'].items() if k != 'horizon'})

display(dlinear_tuning_df)
print('Best DLinear config:', dlinear_best['config'])
print(f"Best DLinear validation WMAE: {dlinear_best['wmae']:,.2f}")


## Pipeline for Best Hyperparameters


In [ ]:
class DLinearInferencePipeline(mlflow.pyfunc.PythonModel):
    def __init__(
        self,
        model,
        history_df,
        norm_stats,
        config,
        future_feature_cols,
        origin_date,
        feature_table,
    ):
        self.model = copy.deepcopy(model).to(torch.device('cpu')).eval()
        self.history_df = history_df[
            ['Store', 'Dept', 'Date', TARGET]
        ].copy()
        self.norm_stats = norm_stats.copy()
        self.config = config.copy()
        self.future_feature_cols = list(future_feature_cols)
        self.origin_date = pd.Timestamp(origin_date)
        self.feature_table = feature_table.copy()

    def predict(self, context, model_input, params=None):
        raw = model_input.copy().reset_index(drop=True)
        raw['Date'] = pd.to_datetime(raw['Date'])
        raw['_DLinear_Input_Order'] = np.arange(len(raw))

        markdown_cols = [
            'MarkDown1', 'MarkDown2', 'MarkDown3',
            'MarkDown4', 'MarkDown5',
        ]
        if not set(markdown_cols).issubset(raw.columns):
            feature_table = self.feature_table.copy()
            feature_table['Date'] = pd.to_datetime(feature_table['Date'])
            raw = raw.merge(
                feature_table,
                on=['Store', 'Date', 'IsHoliday'],
                how='left',
                sort=False,
                validate='many_to_one',
            )

        prepared = prepare_dlinear_frame(raw)
        predictions, _ = predict_dlinear_origin(
            self.model,
            self.history_df,
            prepared,
            self.norm_stats,
            int(self.config['lookback']),
            int(self.config['horizon']),
            self.origin_date,
            torch.device('cpu'),
            self.future_feature_cols,
            batch_size=512,
            pin_memory=False,
        )

        output = pd.DataFrame({
            '_DLinear_Input_Order': prepared['_DLinear_Input_Order'].to_numpy(),
            'Weekly_Sales': predictions,
        })
        output = output.sort_values('_DLinear_Input_Order')
        return output[['Weekly_Sales']].reset_index(drop=True)

if 'dlinear_best' not in globals() or not dlinear_best.get('config'):
    raise RuntimeError('Run the DLinear_Tuning cell first.')

final_cfg = dlinear_best['config'].copy()
final_epochs = max(1, int(dlinear_tuning_df.iloc[0]['best_epoch']))
final_cfg['epochs'] = final_epochs
final_cfg['patience'] = final_epochs + 1

final_stats = dlinear_norm_stats(dlinear_train)
final_ds = DLinearWindowDataset(
    dlinear_train,
    final_cfg['lookback'],
    final_cfg['horizon'],
    final_stats,
    DLINEAR_FUTURE_FEATURES,
)
final_loader = DataLoader(
    final_ds,
    batch_size=DLINEAR_BATCH_SIZE,
    shuffle=True,
    num_workers=DLINEAR_NUM_WORKERS,
    pin_memory=DLINEAR_PIN_MEMORY,
)

final_future_feature_count = (
    len(DLINEAR_FUTURE_FEATURES)
    if final_cfg.get('use_future_features', True)
    else 0
)
final_model = CleanDLinear(
    final_cfg['lookback'],
    final_cfg['horizon'],
    kernel_size=final_cfg['kernel'],
    future_feature_count=final_future_feature_count,
).to(DLINEAR_DEVICE)
final_optimizer = torch.optim.AdamW(
    final_model.parameters(),
    lr=final_cfg['lr'],
    weight_decay=final_cfg.get('weight_decay', 1e-4),
)

final_losses = []
for epoch in range(1, final_epochs + 1):
    loss = train_one_dlinear_epoch(
        final_model,
        final_loader,
        final_optimizer,
        DLINEAR_DEVICE,
    )
    final_losses.append(loss)
    if epoch == 1 or epoch % 5 == 0:
        print(f'final epoch={epoch:03d} train_loss={loss:.5f}')

dlinear_pipeline = DLinearInferencePipeline(
    model=final_model,
    history_df=dlinear_train,
    norm_stats=final_stats,
    config=final_cfg,
    future_feature_cols=DLINEAR_FUTURE_FEATURES,
    origin_date=DLINEAR_TEST_ORIGIN,
    feature_table=features_raw,
)

pipeline_output = dlinear_pipeline.predict(None, test_raw)
pipeline_predictions = pipeline_output['Weekly_Sales'].to_numpy()

assert len(pipeline_predictions) == len(test_raw)
assert np.isfinite(pipeline_predictions).all()
assert (pipeline_predictions >= 0).all()

DLINEAR_CHECKPOINT_PATH = 'dlinear_final.pt'

torch.save({
    'model_state_dict': final_model.state_dict(),
    'config': final_cfg,
    'norm_stats': final_stats,
    'lookback': final_cfg['lookback'],
    'horizon': final_cfg['horizon'],
    'origin_date': DLINEAR_TEST_ORIGIN,
    'future_features': DLINEAR_FUTURE_FEATURES,
}, DLINEAR_CHECKPOINT_PATH)

with mlflow.start_run(run_name='DLinear_Final') as run:
    mlflow.log_params({
        **{f'final_{k}': v for k, v in final_cfg.items()},
        'device': str(DLINEAR_DEVICE),
        'final_train_rows': len(dlinear_train),
        'test_rows': len(test_raw),
        'reference_val_wmae': float(dlinear_best['wmae']),
        'registered_model_name': 'WalmartSales_DLinear',
        'selection_source': 'DLinear_Tuning',
        'feature_contract': 'raw_merged_to_weekly_sales',
        'pipeline_preserves_input_order': True,
    })
    mlflow.log_metric('selection_wmae', float(dlinear_best['wmae']))
    mlflow.set_tags({
        'pipeline_contract': 'raw_merged_dataframe_to_weekly_sales',
        'winner_selected_by': 'minimum_validation_wmae',
    })
    mlflow.log_artifact(DLINEAR_CHECKPOINT_PATH)
    mlflow.pyfunc.log_model(
        name='dlinear_pipeline',
        python_model=dlinear_pipeline,
        registered_model_name='WalmartSales_DLinear',
    )

print(f'Pipeline check rows: {len(pipeline_predictions)}')
print(f'Run ID: {run.info.run_id}')
print('Model registered as: WalmartSales_DLinear')
